# 📖 Notebook 1: Order Matching Engine

Robinhood is a **brokerage**, not an exchange. But to understand how orders get filled,
we need to understand what happens on the exchange side. In this notebook we build a
simplified order matching engine so you can see the mechanics first-hand.

## Learning Objectives

By the end of this notebook you will understand:
- The difference between **market orders** and **limit orders**
- How an **order book** works (bids and asks)
- How a matching engine pairs buyers with sellers
- The order lifecycle inside a brokerage: `pending → submitted → filled`
- Why **consistency** matters and how failures are handled

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/robinhood
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `robinhood_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json
import time
import uuid
from datetime import datetime

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "robinhood_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

# Quick connection test
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM symbols")
    print(f"✅ Connected to PostgreSQL — {cur.fetchone()[0]} symbols loaded")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

## 📚 Background: How Stock Trading Works

Before we write code, let's understand the key concepts:

### Market Orders vs Limit Orders

| Type | What It Means | Example |
|------|--------------|----------|
| **Market** | "Buy/sell NOW at whatever the current price is" | "Buy 10 shares of AAPL at market price" |
| **Limit**  | "Buy/sell ONLY at this price or better" | "Buy 10 AAPL only if price ≤ $190" |

**Market orders** execute immediately but you don't control the price.  
**Limit orders** give you price control but might never execute if the price never reaches your target.

### The Order Book

An exchange keeps an **order book** — a sorted list of all outstanding limit orders:

```
         ORDER BOOK for AAPL
  ┌──────────────────────────────┐
  │  ASKS (sellers)   ↑ price    │
  │  $192.00  ×  50 shares       │
  │  $191.50  × 100 shares       │  ← lowest ask (best price to buy)
  │ ─────────── SPREAD ───────── │
  │  $191.00  × 200 shares       │  ← highest bid (best price to sell)
  │  $190.50  ×  75 shares       │
  │  BIDS (buyers)    ↓ price    │
  └──────────────────────────────┘
```

The **spread** is the gap between the highest bid and lowest ask.  
When a market buy comes in, it matches against the lowest ask.

## 🏗️ Building a Simple Order Book

Let's build an in-memory order book. This simulates what happens *inside* an exchange.

In [ ]:
import heapq
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class LimitOrder:
    """A single limit order sitting on the book."""
    order_id: str
    side: str                  # 'buy' or 'sell'
    price_cents: int           # price in cents to avoid float issues
    quantity: int              # shares remaining
    timestamp: float = field(default_factory=time.time)


class OrderBook:
    """
    A simplified order book for one symbol.
    
    Bids (buy orders)  are stored in a max-heap (highest price first).
    Asks (sell orders) are stored in a min-heap (lowest price first).
    
    We use Python's heapq (min-heap) and negate prices for the bid side.
    """

    def __init__(self, symbol: str):
        self.symbol = symbol
        self.bids: list = []   # max-heap via negated prices
        self.asks: list = []   # min-heap
        self.trades: list = [] # executed trades log

    def add_limit_order(self, order: LimitOrder):
        """Place a limit order on the book, matching if possible."""
        remaining = order.quantity

        if order.side == 'buy':
            # Try to match against existing asks (sellers)
            while remaining > 0 and self.asks:
                best_ask_price, _, best_ask = self.asks[0]
                if order.price_cents < best_ask_price:
                    break  # our bid is too low
                # Match!
                heapq.heappop(self.asks)
                matched_qty = min(remaining, best_ask.quantity)
                self._record_trade(order, best_ask, best_ask_price, matched_qty)
                remaining -= matched_qty
                best_ask.quantity -= matched_qty
                if best_ask.quantity > 0:
                    heapq.heappush(self.asks, (best_ask.price_cents, best_ask.timestamp, best_ask))

            # If shares remain, rest on the book
            if remaining > 0:
                order.quantity = remaining
                heapq.heappush(self.bids, (-order.price_cents, order.timestamp, order))

        else:  # sell
            while remaining > 0 and self.bids:
                neg_best_bid, _, best_bid = self.bids[0]
                best_bid_price = -neg_best_bid
                if order.price_cents > best_bid_price:
                    break  # our ask is too high
                heapq.heappop(self.bids)
                matched_qty = min(remaining, best_bid.quantity)
                self._record_trade(best_bid, order, best_bid_price, matched_qty)
                remaining -= matched_qty
                best_bid.quantity -= matched_qty
                if best_bid.quantity > 0:
                    heapq.heappush(self.bids, (-best_bid.price_cents, best_bid.timestamp, best_bid))

            if remaining > 0:
                order.quantity = remaining
                heapq.heappush(self.asks, (order.price_cents, order.timestamp, order))

    def add_market_order(self, side: str, quantity: int) -> list:
        """Execute a market order — matches immediately at best available price."""
        fills = []
        remaining = quantity

        if side == 'buy':
            while remaining > 0 and self.asks:
                best_price, _, best_ask = heapq.heappop(self.asks)
                matched_qty = min(remaining, best_ask.quantity)
                fills.append((best_price, matched_qty))
                remaining -= matched_qty
                best_ask.quantity -= matched_qty
                if best_ask.quantity > 0:
                    heapq.heappush(self.asks, (best_ask.price_cents, best_ask.timestamp, best_ask))
        else:
            while remaining > 0 and self.bids:
                neg_price, _, best_bid = heapq.heappop(self.bids)
                matched_qty = min(remaining, best_bid.quantity)
                fills.append((-neg_price, matched_qty))
                remaining -= matched_qty
                best_bid.quantity -= matched_qty
                if best_bid.quantity > 0:
                    heapq.heappush(self.bids, (-best_bid.price_cents, best_bid.timestamp, best_bid))

        return fills

    def _record_trade(self, buyer, seller, price_cents, quantity):
        self.trades.append({
            "buyer": buyer.order_id,
            "seller": seller.order_id,
            "price_cents": price_cents,
            "quantity": quantity,
            "time": time.time()
        })

    def display(self):
        """Pretty-print the current order book."""
        print(f"\n📖 Order Book: {self.symbol}")
        print("=" * 45)

        # Asks (sorted lowest first, display highest first)
        ask_list = sorted([(p, o) for p, _, o in self.asks], reverse=True)
        print("  ASKS (sellers)")
        if not ask_list:
            print("    (empty)")
        for price, order in ask_list:
            print(f"    ${price/100:>10.2f}  ×  {order.quantity:>4} shares")

        print("  ─────────── SPREAD ───────────")

        bid_list = sorted([(-p, o) for p, _, o in self.bids], reverse=True)
        if not bid_list:
            print("    (empty)")
        for price, order in bid_list:
            print(f"    ${price/100:>10.2f}  ×  {order.quantity:>4} shares")
        print("  BIDS (buyers)")
        print()

In [ ]:
# Let's build an order book for AAPL and add some limit orders

book = OrderBook("AAPL")

# Sellers willing to sell at these prices
book.add_limit_order(LimitOrder("S1", "sell", 19200, 50))
book.add_limit_order(LimitOrder("S2", "sell", 19150, 100))
book.add_limit_order(LimitOrder("S3", "sell", 19300, 30))

# Buyers willing to buy at these prices
book.add_limit_order(LimitOrder("B1", "buy", 19100, 200))
book.add_limit_order(LimitOrder("B2", "buy", 19050, 75))
book.add_limit_order(LimitOrder("B3", "buy", 19000, 150))

book.display()

print("💡 Notice the spread: sellers want ≥$191.50, buyers offer ≤$191.00")
print("   No trade happens yet because nobody agrees on a price.")

In [ ]:
# Now a buyer places a limit order at $191.50 — it crosses the spread!

print("🔔 New order: BUY 60 AAPL @ $191.50 (limit)")
print()

book.add_limit_order(LimitOrder("B4", "buy", 19150, 60))

# Show what happened
print("Trades executed:")
for t in book.trades:
    print(f"  {t['buyer']} bought {t['quantity']} shares "
          f"from {t['seller']} @ ${t['price_cents']/100:.2f}")

book.display()

print("💡 The buyer's order matched against S2 (the lowest ask at $191.50).")
print("   S2 had 100 shares, buyer only wanted 60, so 40 shares remain on the ask side.")

In [ ]:
# Now let's try a market order — it takes the best available price

print("🔔 New order: MARKET BUY 50 AAPL")
print()

fills = book.add_market_order("buy", 50)

total_cost = sum(price * qty for price, qty in fills)
total_shares = sum(qty for _, qty in fills)
avg_price = total_cost / total_shares if total_shares else 0

print("Fills:")
for price, qty in fills:
    print(f"  {qty} shares @ ${price/100:.2f}")

print(f"\nTotal: {total_shares} shares, avg price ${avg_price/100:.2f}")
print(f"Total cost: ${total_cost/100:.2f}")

book.display()

print("💡 The market order ate through the best asks.")
print("   If the book is thin (few shares), a big market order can move the price a lot!")
print("   This is called 'slippage'.")

## 🏦 Brokerage Order Lifecycle

Now let's see the **brokerage side** — how Robinhood handles an order from the user
through to the exchange and back.

```
  User App          Robinhood Backend         Exchange
  ───────          ─────────────────         ────────
    │                     │                      │
    │  POST /order        │                      │
    │ ──────────────────► │                      │
    │                     │  1. Save order        │
    │                     │     status=pending    │
    │                     │                      │
    │                     │  2. Submit to         │
    │                     │     exchange ────────►│
    │                     │                      │
    │                     │  3. Get external ID  │
    │                     │  ◄───────────────────│
    │                     │     status=submitted  │
    │                     │                      │
    │  {order: submitted} │                      │
    │ ◄────────────────── │                      │
    │                     │                      │
    │                     │  4. Trade feed:       │
    │                     │     order filled     │
    │                     │  ◄───────────────────│
    │                     │     status=filled     │
    │                     │                      │
    │  {order: filled}    │                      │
    │ ◄────────────────── │                      │
```

**Key insight**: We save to our database *before* talking to the exchange.  
This way, if anything fails mid-way, we have a record and can recover.

In [ ]:
# Simulate the full brokerage order lifecycle against our Postgres database

import random

def simulate_exchange_submit(order_id, symbol, side, qty, price_cents):
    """Pretend to talk to an exchange. Returns an external order ID."""
    time.sleep(0.05)  # simulate network latency
    return f"EX-{uuid.uuid4().hex[:8]}"

def create_order(user_id: int, ticker: str, side: str, order_type: str,
                 quantity: int, limit_price_cents: int = None):
    """
    Full brokerage order flow:
    1. Save order as 'pending'
    2. Submit to exchange
    3. Update to 'submitted' with external ID
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Look up the symbol
    cur.execute("SELECT id, last_price_cents FROM symbols WHERE ticker = %s", (ticker,))
    symbol = cur.fetchone()
    if not symbol:
        print(f"❌ Unknown ticker: {ticker}")
        conn.close()
        return None

    # Step 1: Save order as PENDING
    cur.execute("""
        INSERT INTO orders (user_id, symbol_id, side, order_type, quantity, limit_price_cents, status)
        VALUES (%s, %s, %s, %s, %s, %s, 'pending')
        RETURNING id, status, created_at
    """, (user_id, symbol['id'], side, order_type, quantity, limit_price_cents))
    order = cur.fetchone()
    print(f"📝 Step 1: Order #{order['id']} saved as PENDING")

    # Step 2: Submit to exchange
    price = limit_price_cents or symbol['last_price_cents']
    try:
        ext_id = simulate_exchange_submit(order['id'], ticker, side, quantity, price)
        print(f"📡 Step 2: Submitted to exchange → external ID: {ext_id}")
    except Exception as e:
        # If exchange fails, mark order as failed
        cur.execute("UPDATE orders SET status = 'failed' WHERE id = %s", (order['id'],))
        print(f"❌ Step 2: Exchange submission failed: {e}")
        conn.close()
        return None

    # Step 3: Update order with external ID and status
    cur.execute("""
        UPDATE orders SET status = 'submitted', external_order_id = %s, updated_at = NOW()
        WHERE id = %s
        RETURNING *
    """, (ext_id, order['id']))
    updated = cur.fetchone()
    print(f"✅ Step 3: Order #{updated['id']} updated to SUBMITTED")

    conn.close()
    return dict(updated)


# Create a market buy order for Alice (user 1)
print("=" * 60)
print("Alice wants to BUY 10 shares of GOOGL at market price")
print("=" * 60)
order = create_order(user_id=1, ticker="GOOGL", side="buy",
                     order_type="market", quantity=10)
print()
print(f"Order result: {order['side'].upper()} {order['quantity']} shares, "
      f"status={order['status']}")

In [ ]:
# Now simulate the exchange filling the order (trade feed callback)

def simulate_trade_fill(order_id: int, fill_price_cents: int):
    """
    Simulate receiving a fill from the exchange's trade feed.
    Updates order status and records the trade.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Look up the order
    cur.execute("SELECT * FROM orders WHERE id = %s", (order_id,))
    order = cur.fetchone()

    if not order:
        print(f"❌ Order #{order_id} not found")
        conn.close()
        return

    # Record the trade
    cur.execute("""
        INSERT INTO trades (order_id, symbol_id, price_cents, quantity)
        VALUES (%s, %s, %s, %s)
    """, (order['id'], order['symbol_id'], fill_price_cents, order['quantity']))

    # Update order to filled
    cur.execute("""
        UPDATE orders
        SET status = 'filled',
            filled_quantity = quantity,
            filled_avg_price_cents = %s,
            updated_at = NOW()
        WHERE id = %s
        RETURNING *
    """, (fill_price_cents, order['id']))
    updated = cur.fetchone()

    print(f"✅ Order #{updated['id']} FILLED: {updated['filled_quantity']} shares "
          f"@ ${fill_price_cents/100:.2f}")

    conn.close()
    return dict(updated)


# Simulate the fill
print("📡 Exchange trade feed: order filled!")
print()
filled = simulate_trade_fill(order['id'], fill_price_cents=17830)

In [ ]:
# View all orders for Alice

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT o.id, s.ticker, o.side, o.order_type, o.quantity,
           o.status, o.filled_quantity,
           o.filled_avg_price_cents / 100.0 AS filled_price,
           o.created_at
    FROM orders o
    JOIN symbols s ON o.symbol_id = s.id
    WHERE o.user_id = 1
    ORDER BY o.created_at DESC
""")

print("📋 Alice's Orders")
print("=" * 90)
print(f"{'ID':>4} {'Ticker':<6} {'Side':<5} {'Type':<7} {'Qty':>5} {'Status':<12} "
      f"{'Filled':>6} {'Price':>10}")
print("-" * 90)

for row in cur.fetchall():
    price_str = f"${row['filled_price']:.2f}" if row['filled_price'] else "-"
    print(f"{row['id']:>4} {row['ticker']:<6} {row['side']:<5} {row['order_type']:<7} "
          f"{row['quantity']:>5} {row['status']:<12} {row['filled_quantity']:>6} {price_str:>10}")

conn.close()

## 🛡️ Handling Failures: Why Order Consistency Matters

What happens if the system crashes *between* submitting to the exchange and updating our DB?

```
  Failure Points in the Order Flow:
  
  1. Save order (pending)    ← Failure here? Easy: tell user it failed.
  2. Submit to exchange       ← Failure here? Mark as failed, done.
  3. Update DB (submitted)    ← Failure here? 😱 Exchange has our order
                                 but our DB still says 'pending'!
```

**Solution**: A cleanup job scans for orders stuck in `pending` status and checks
the exchange to see if they actually went through. Let's simulate this.

In [ ]:
# Simulate a failure: order submitted to exchange but DB update crashed

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Create an order that "got stuck" in pending
cur.execute("""
    INSERT INTO orders (user_id, symbol_id, side, order_type, quantity, status, created_at)
    VALUES (2, 1, 'buy', 'market', 5, 'pending', NOW() - INTERVAL '5 minutes')
    RETURNING id
""")
stuck_id = cur.fetchone()['id']
print(f"⚠️  Simulated stuck order #{stuck_id} — status is 'pending' but exchange may have it")


def cleanup_stuck_orders(max_age_minutes: int = 2):
    """
    Background job that finds orders stuck in 'pending' for too long.
    In production, this would query the exchange to check the real status.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT id, user_id, symbol_id, side, quantity
        FROM orders
        WHERE status = 'pending'
          AND created_at < NOW() - INTERVAL '%s minutes'
    """ % max_age_minutes)

    stuck = cur.fetchall()
    print(f"🔍 Cleanup job found {len(stuck)} stuck order(s)")

    for order in stuck:
        # In reality: query exchange API with clientOrderId to check status
        exchange_says_filled = random.choice([True, False])

        if exchange_says_filled:
            cur.execute("""
                UPDATE orders SET status = 'filled', filled_quantity = quantity,
                       filled_avg_price_cents = 19000, updated_at = NOW()
                WHERE id = %s
            """, (order['id'],))
            print(f"  ✅ Order #{order['id']} was actually filled on exchange → marked filled")
        else:
            cur.execute("""
                UPDATE orders SET status = 'failed', updated_at = NOW()
                WHERE id = %s
            """, (order['id'],))
            print(f"  ❌ Order #{order['id']} not found on exchange → marked failed")

    conn.close()

cleanup_stuck_orders()

print()
print("💡 This cleanup pattern ensures eventual consistency.")
print("   Even if our system crashes, no order is lost or duplicated.")

## 🔄 Cancel Flow

Cancelling an order follows a similar pattern with its own safety steps:

1. Mark order as `pending_cancel` (so cleanup jobs know what we intended)
2. Send cancel request to exchange
3. Update order to `cancelled`

If step 3 fails, the cleanup job picks up `pending_cancel` orders and verifies with the exchange.

In [ ]:
# Create a limit order, then cancel it

print("1️⃣  Create a limit order")
limit_order = create_order(user_id=1, ticker="TSLA", side="buy",
                           order_type="limit", quantity=25,
                           limit_price_cents=24000)
print()

def cancel_order(order_id: int):
    """Cancel an outstanding order."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Step 1: Mark as pending_cancel
    cur.execute("""
        UPDATE orders SET status = 'pending_cancel', updated_at = NOW()
        WHERE id = %s AND status IN ('pending', 'submitted')
        RETURNING *
    """, (order_id,))
    order = cur.fetchone()

    if not order:
        print(f"❌ Order #{order_id} cannot be cancelled (already filled or cancelled)")
        conn.close()
        return None

    print(f"📝 Step 1: Order #{order_id} marked as PENDING_CANCEL")

    # Step 2: Send cancel to exchange
    time.sleep(0.03)  # simulate network
    print(f"📡 Step 2: Cancel sent to exchange")

    # Step 3: Update status
    cur.execute("""
        UPDATE orders SET status = 'cancelled', updated_at = NOW()
        WHERE id = %s RETURNING *
    """, (order_id,))
    cancelled = cur.fetchone()
    print(f"✅ Step 3: Order #{order_id} CANCELLED")

    conn.close()
    return dict(cancelled)

print("2️⃣  Cancel the order")
cancel_order(limit_order['id'])

## 🧹 Cleanup

In [ ]:
# Remove the test orders we created during this notebook
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM trades WHERE order_id > 5")
cur.execute("DELETE FROM orders WHERE id > 5")
print(f"🧹 Cleaned up test orders and trades")
conn.close()

## 📚 Summary

### Key Takeaways

1. **Order Book** — a sorted list of bids (buy) and asks (sell). Matching happens when prices cross.
2. **Market orders** execute immediately at the best available price; **limit orders** wait for a target price.
3. **Robinhood is a brokerage**, not an exchange — it routes orders to exchanges and tracks them.
4. **Order lifecycle**: `pending → submitted → filled/cancelled/failed`.
5. **Save to DB before sending to exchange** — this is crucial for crash recovery.
6. **Cleanup jobs** ensure eventual consistency by reconciling stuck orders.

### For System Design Interviews

- Always mention the order status state machine
- Explain why you save to DB first (fault tolerance)
- Mention cleanup/reconciliation jobs for eventual consistency
- Use cents (integers) for prices — never floating point for money!

### Next Up

In **Notebook 2**, we'll track **portfolios and positions** — how buying and selling
updates a user's holdings and profit/loss.